# ST554_Final Project

Final project for ST554: Analysis of Big Data at NC State University, Spring 2026

Creator: Cole Hammett

Purpose: Predicting power consumption (Zone 3) for Tetouan City using PySpark MLlib and Structured Streaming.

Date: 30th of April, 2026

Instructor: Justin Post

# 1. Setup & Data Loading

In [1]:
# Import necessary modules

import pandas as pd
from pyspark.sql import SparkSession

## 1.1 Starting a SparkSession

`local[*]` tells Spark to run locally using all available CPU cores.

`appName` is just a label for the UI/logs.

In [2]:
# Create a SparkSession using all available local cores
spark = SparkSession.builder \
    .master('local[*]') \
    .appName('power_zone_prediction') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/30 11:16:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 1.2 Reading data with pandas

Two-step load, read in .csv with pandas THEN convert rather than reading directly into Spark

In [3]:
# Read power_ml_data.csv directly from the course URL into a pandas DataFrame
power_pd = pd.read_csv("https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv")

In [4]:
# Quick sanity checks on the pandas DataFrame
print("Shape:", power_pd.shape)
print("\nFirst few rows:")
power_pd.head()

Shape: (47174, 10)

First few rows:


,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0


## 1.3 Converting to a Spark Dataframe

In [5]:
# spark.createDataFrame() accepts a pandas DataFrame directly
spark_df = spark.createDataFrame(power_pd)

In [6]:
# Verify the schema for catching type issues before pipeline work
# In particular, we'll check whether Hour is already a DoubleType rather than `LongType` or `IntegerType`
print("Spark DataFrame schema:")
spark_df.printSchema()

Spark DataFrame schema:
root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)



In [7]:
# Preview the first few rows of the Spark DataFrame
spark_df.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
+-----------+--------+----------+-------